# BAFU import

### 1. Import required packages

In [ ]:
import bw2data as bd
import bw2io as bi

from functions.fun_bafu_import import (
    apply_elementary_flow_mapping,
    load_bafu_ecospold,
    load_elementary_flow_mapping,
)
from functions.utils import Config

### 2. Select project

In [ ]:
bd.projects.set_current(Config.PROJECT_NAME)

### 3. Import biosphere database

In [ ]:
if "biosphere3" not in bd.databases:
    bi.create_default_biosphere3()

### 4. Convert the BAFU EcoSpold files

BAFU 2025 is provided in EcoSpold 1 format. The helper function below reads the XML files and converts them into Brightway-style dataset dictionaries.

The detailed parsing code, including unit conversion, uncertainty handling, metadata extraction, and exchange classification, is available in `functions/fun_bafu_import.py`.

In [ ]:
all_data = load_bafu_ecospold(
    Config.BAFU_ECOSPOLD_PATH,
    Config.BAFU_DATABASE_NAME,
)

### 5. Map BAFU elementary flows

The supplied mapping file translates BAFU elementary-flow names and categories to the corresponding names and categories used by the Brightway biosphere database.

In [ ]:
mapping = load_elementary_flow_mapping(Config.BAFU_MAPPING_FILE_PATH)
all_data = apply_elementary_flow_mapping(all_data, mapping)

### 6. Create Brightway importer

The converted BAFU datasets are assigned to a Brightway LCI importer and the standard import strategies are applied.

In [ ]:
importer = bi.importers.base_lci.LCIImporter(
    db_name=Config.BAFU_DATABASE_NAME
)
importer.data = all_data
importer.apply_strategies()


### 7. Link technosphere and biosphere exchanges

Technosphere exchanges are linked to other BAFU activities using activity name, reference product, and location. Biosphere exchanges are linked to the configured BAFU biosphere database using name and categories.

In [ ]:
importer.match_database(
    fields=["name", "reference product", "location"]
)
importer.match_database(
    Config.BAFU_BIOSPHERE_NAME,
    fields=["name", "categories"],
)

### 8. Inspect import statistics

Review any unlinked exchanges before writing the database.

In [ ]:
# Inspect unlinked exchanges
pd.DataFrame(importer.unlinked)

### 9. Handle remaining unlinked exchanges

In [ ]:
importer.drop_unlinked(i_am_reckless=True)

### 10. Write the BAFU database

This writes the processed datasets to the current Brightway project.

In [ ]:
importer.write_database()

### 11. Verify the import

The BAFU database should now appear in the list of databases in the current Brightway project.

In [ ]:
bd.databases